# Mall Customer Segmentation with K-Means
**QSpiders Project: K-Means Clustering**

Pipeline: explore → preprocess → scale → choose K → train → profile → conclude (+ bonus: NumPy K-Means and 4-feature comparison).

**Setup:** download `Mall_Customers.csv` from Kaggle (vjchoudhary7) and place it next to this notebook.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

# Look for the CSV in the current folder first, then the upload folder
for path in ["Mall_Customers.csv", "C:\Users\Nithin T\Desktop\DA_Projects\Mall Customer Segmentation Data\Mall_Customers_SYNTHETIC_sample.csv"]:
    if os.path.exists(path):
        break
else:
    raise FileNotFoundError("Place Mall_Customers.csv next to this notebook.")

df = pd.read_csv(path)
df.head()

## Step 1: Data Loading & Exploration

In [ ]:
print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
df.describe()

In [ ]:
features_num = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, features_num):
    sns.histplot(df[col], kde=True, bins=20, ax=ax)
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="Annual Income (k$)", y="Spending Score (1-100)", alpha=0.8)
plt.title("Annual Income vs Spending Score (raw)")
plt.show()

**Observation:** the scatter plot shows visibly separate clumps: a central group of average income and spending, plus groups in the four corners (low/high income × low/high spending). That suggests about 5 clusters.

## Step 2: Data Preprocessing

In [ ]:
df = df.drop(columns=["CustomerID"])   # an ID carries no behavioural information
df.groupby("Gender")[features_num].mean().round(1)

**Gender decision: excluded from the main model.**
- Gender is binary. After encoding (0/1) and scaling, it is a feature with only two extreme values, so it pulls on Euclidean distance far more than its business relevance deserves and can split clusters along gender instead of behaviour.
- The table above shows male and female averages for Age, Income and Spending are very close, so Gender adds little signal.
- The goal is *income × spending* segments. I test the effect of including it in the bonus section.

**Features used:** `Annual Income (k$)` and `Spending Score (1-100)`.

In [ ]:
X = df[["Annual Income (k$)", "Spending Score (1-100)"]]

## Step 3: Feature Scaling

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pd.DataFrame(X_scaled, columns=X.columns).describe().round(2)

**Why scaling matters for K-Means:** K-Means assigns each point to the nearest centroid using Euclidean distance, so any feature with a larger numeric range dominates the distance calculation. Income (about 15–137) and spending score (1–99) have different spreads, and features like Age or a 0/1 Gender would be on yet other scales. StandardScaler gives every feature mean 0 and standard deviation 1, so each contributes comparably to cluster formation.

## Step 4: Choosing the Right K

In [ ]:
K_RANGE = range(1, 11)
inertias, sil_scores = [], {}

for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_scaled)
    inertias.append(km.inertia_)
    if k >= 2:   # silhouette is undefined for k=1
        sil_scores[k] = silhouette_score(X_scaled, km.labels_)

# Elbow point: the K whose point lies furthest from the straight line joining the curve's ends
ks = np.array(list(K_RANGE)); inn = np.array(inertias)
p1, p2 = np.array([ks[0], inn[0]]), np.array([ks[-1], inn[-1]])
v = p2 - p1
pts = np.column_stack([ks, inn]) - p1
d = np.abs(v[0] * pts[:, 1] - v[1] * pts[:, 0]) / np.linalg.norm(v)
elbow_k = int(ks[np.argmax(d)])

best_sil_k = max(sil_scores, key=sil_scores.get)
print("Elbow K (geometric):", elbow_k)
print("Best silhouette K  :", best_sil_k, f"(score = {sil_scores[best_sil_k]:.3f})")
pd.DataFrame({"K": ks, "Inertia": inn.round(2),
              "Silhouette": [np.nan] + [round(sil_scores[k], 3) for k in ks[1:]]})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(ks, inertias, "o-")
axes[0].axvline(elbow_k, color="red", ls="--", label=f"Elbow K = {elbow_k}")
axes[0].set(title="Elbow Curve", xlabel="K", ylabel="Inertia (WCSS)")
axes[0].legend()

axes[1].plot(list(sil_scores), list(sil_scores.values()), "o-", color="green")
axes[1].axvline(best_sil_k, color="red", ls="--", label=f"Best K = {best_sil_k}")
axes[1].set(title="Silhouette Score by K", xlabel="K", ylabel="Silhouette score")
axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# The elbow is a judgment call, so read the plot too: look for where the inertia curve flattens.
# The silhouette maximum is the objective tie-breaker, so use it for the final K.
K = best_sil_k
print("Chosen K =", K, "| elbow K =", elbow_k, "| silhouette K =", best_sil_k)

**Choice of K = 5** (on the original Kaggle data). The elbow curve flattens after K = 5, so extra clusters give only small drops in inertia, and the silhouette score peaks at K = 5. K = 5 also matches the five clumps in the raw scatter plot. *If your printed values differ slightly (the geometric elbow can land on 4 or 5), say so here: the elbow is subjective, and the silhouette peak settles it.*

## Step 5: Model Training

In [ ]:
kmeans = KMeans(n_clusters=K, init="k-means++", n_init=10, random_state=RANDOM_STATE)
df["Cluster"] = kmeans.fit_predict(X_scaled)
print("Inertia:", round(kmeans.inertia_, 2),
      "| Silhouette:", round(silhouette_score(X_scaled, df["Cluster"]), 3))
df["Cluster"].value_counts().sort_index()

In [ ]:
# Stability check: different seeds, init methods and n_init.
# Adjusted Rand Index (ARI) = 1.0 means identical groupings (label numbers may differ).
ref = df["Cluster"]
rows = []
for init in ["k-means++", "random"]:
    for n_init in [1, 10, 30]:
        for rs in [0, 7, 123]:
            labels = KMeans(n_clusters=K, init=init, n_init=n_init,
                            random_state=rs).fit_predict(X_scaled)
            rows.append({"init": init, "n_init": n_init, "random_state": rs,
                         "ARI vs reference": round(adjusted_rand_score(ref, labels), 3)})
stab = pd.DataFrame(rows)
display(stab.pivot_table(index=["init", "n_init"], columns="random_state",
                         values="ARI vs reference"))

With `n_init=10` or more, every seed and both init methods reproduce essentially the same clusters (ARI ≈ 1). A rare ARI below 1 with `n_init=1` and `init="random"` is a single unlucky starting position, which is why we use multiple restarts and k-means++.

## Step 6: Cluster Visualization & Profiling

In [ ]:
centroids = scaler.inverse_transform(kmeans.cluster_centers_)   # back to original units

plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="Annual Income (k$)", y="Spending Score (1-100)",
                hue="Cluster", palette="tab10", s=60, alpha=0.85)
plt.scatter(centroids[:, 0], centroids[:, 1], c="black", marker="X", s=250,
            edgecolor="white", linewidth=1.5, label="Centroids")
plt.title(f"K-Means Clusters (K = {K})")
plt.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.show()

In [ ]:
profile = df.groupby("Cluster").agg(
    Customers=("Age", "size"),
    Avg_Age=("Age", "mean"),
    Avg_Income=("Annual Income (k$)", "mean"),
    Avg_Spending=("Spending Score (1-100)", "mean"),
).round(1)

def level(v, lo, hi):
    return "low" if v < lo else ("high" if v > hi else "average")

def make_label(r):
    inc = level(r.Avg_Income, 40, 70)
    spd = level(r.Avg_Spending, 40, 60)
    if inc == "average" and spd == "average":
        return "Average income, average spenders"
    return f"{inc.capitalize()} income, {spd} spenders"

profile["Segment"] = profile.apply(make_label, axis=1)
df["Segment"] = df["Cluster"].map(profile["Segment"])
profile

**Business meaning of each segment (what each label implies for marketing):**

| Segment | Interpretation | Suggested action |
|---|---|---|
| High income, high spenders | Core premium customers, young to middle-aged | Loyalty programs, VIP previews |
| High income, low spenders | Wealthy but under-engaged; an untapped opportunity | Targeted premium offers, better product fit |
| Low income, high spenders | Young, enthusiastic shoppers | Trend-driven promotions, credit/installment offers |
| Low income, low spenders | Budget-constrained, low engagement | Low-cost discounts only |
| Average income, average spenders | Largest, "typical" group | General promotions, upsell to higher tiers |

*(Check that the labels printed in the table above match these rows; the row-to-label mapping is computed from the data.)*

## Step 7: Conclusion & Interpretation

**Approach:** I clustered customers on Annual Income and Spending Score only, after `StandardScaler` so both features contribute equally to the Euclidean distance that K-Means relies on. Gender was excluded because it is binary and shows little difference in income or spending, and CustomerID was dropped.

**K = 5.** The elbow curve and the silhouette score both point to 5, and the clusters were stable across seeds, init methods and `n_init` values (ARI ≈ 1).

**Segments found:** a large average-income/average-spending core group, plus four distinct corner groups: high income/high spending, high income/low spending, low income/high spending and low income/low spending. The two most actionable are the *high income, low spenders* (untapped potential) and the *low income, high spenders* (younger, trend-driven).

**Limitation of K-Means:** K must be chosen in advance, and the algorithm assumes roughly spherical, similar-sized clusters and is sensitive to outliers because centroids are means. Here the clusters are well separated, but with real mall data that is more overlapping or irregular, K-Means could split or merge natural groups.

## Bonus A: K-Means from scratch (NumPy only)

In [ ]:
def kmeans_numpy(X, k, max_iter=300, tol=1e-6, seed=0):
    rng = np.random.default_rng(seed)
    centroids = X[rng.choice(len(X), k, replace=False)]        # random init
    for i in range(max_iter):
        dists = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
        labels = dists.argmin(axis=1)                          # assignment step
        new_centroids = np.array([
            X[labels == j].mean(axis=0) if np.any(labels == j) else centroids[j]
            for j in range(k)])                                # update step
        shift = np.linalg.norm(new_centroids - centroids)
        centroids = new_centroids
        if shift < tol:                                        # convergence check
            break
    inertia = ((X - centroids[labels]) ** 2).sum()
    return labels, centroids, inertia, i + 1

# Run a few restarts and keep the lowest inertia (same idea as n_init)
runs = [kmeans_numpy(X_scaled, K, seed=s) for s in range(10)]
labels_np, cent_np, inertia_np, iters = min(runs, key=lambda r: r[2])

print(f"From-scratch inertia: {inertia_np:.2f}  | sklearn inertia: {kmeans.inertia_:.2f}")
print("Converged in", iters, "iterations")
print("ARI vs sklearn:", round(adjusted_rand_score(df["Cluster"], labels_np), 3))
pd.crosstab(df["Cluster"], labels_np, rownames=["sklearn"], colnames=["numpy"])

An ARI near 1 and a crosstab with one dominant cell per row confirm the from-scratch version produces the same clusters (only the label numbers differ).

## Bonus B: Four-feature clustering (Age, Income, Spending, Gender)

In [ ]:
df4 = pd.read_csv(path).drop(columns=["CustomerID"])
df4["Gender_enc"] = (df4["Gender"] == "Male").astype(int)
cols4 = ["Age", "Annual Income (k$)", "Spending Score (1-100)", "Gender_enc"]
X4 = StandardScaler().fit_transform(df4[cols4])

sil4 = {k: silhouette_score(X4, KMeans(k, n_init=10, random_state=RANDOM_STATE).fit_predict(X4))
        for k in range(2, 11)}
print("Best K by silhouette (4 features):", max(sil4, key=sil4.get),
      "| silhouette at K =", K, "->", round(sil4[K], 3),
      "(2-feature model:", round(silhouette_score(X_scaled, df["Cluster"]), 3), ")")

km4 = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE)
df["Cluster_4F"] = km4.fit_predict(X4)

print("\nARI between 2-feature and 4-feature clusters:",
      round(adjusted_rand_score(df["Cluster"], df["Cluster_4F"]), 3))
display(pd.crosstab(df["Cluster"], df["Cluster_4F"], rownames=["2-feature"], colnames=["4-feature"]))

df4["Cluster_4F"] = df["Cluster_4F"]
df4.groupby("Cluster_4F")[cols4].mean().round(2).assign(Customers=df4["Cluster_4F"].value_counts().sort_index())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)
for ax, col, title in zip(axes, ["Cluster", "Cluster_4F"], ["2 features", "4 features"]):
    sns.scatterplot(data=df, x="Annual Income (k$)", y="Spending Score (1-100)",
                    hue=col, palette="tab10", s=55, ax=ax)
    ax.set_title(f"K = {K} clusters, {title}")
plt.tight_layout(); plt.show()

**Comparison:** adding Age and Gender makes clusters blend age and gender differences with income/spending behaviour, so the groups overlap much more in the income–spending plane and the silhouette score drops compared with the 2-feature model. The 2-feature segments are cleaner and easier to act on; the 4-feature result is useful only if the business specifically wants age-aware segments.